# Phase 2 — FINAL Metrics (self-contained, auto-architecture-matching)

Computes AUC-ROC + bootstrap 95% CIs + confusion matrices + per-class stats for **every** condition that has a saved model, by automatically trying all known architectures against each checkpoint and using whichever (a) loads 100% and (b) reproduces the logged accuracy. Anything that can't be verified is reported transparently, not silently dropped.

Run top to bottom. Set Runtime → GPU. No training (except the optional C3b CV at the end).

## 0. Setup (restart after install)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!pip install -q "monai==1.5.0" nibabel torchio imbalanced-learn scikit-learn
import os; os._exit(0)


## 1. Imports + config

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, json, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchio as tio, pandas as pd
from sklearn.metrics import accuracy_score, matthews_corrcoef, confusion_matrix, roc_auc_score, precision_recall_fscore_support
from monai.networks.nets import SwinUNETR
from tqdm.notebook import tqdm
device=torch.device('cuda'); assert torch.cuda.is_available()

DRIVE='/content/drive/My Drive/ADNI_NewDS/'; RESULTS=os.path.join(DRIVE,'results')
BRAIN_DIR=os.path.join(RESULTS,'processed_mri_scans_brain'); OLD_DIR=os.path.join(RESULTS,'processed_mri_scans_swin')
SPLITS=os.path.join(RESULTS,'patient_id_splits.npz'); CSV=os.path.join(RESULTS,'project_data_cleaned.csv')
BIO=os.path.join(RESULTS,'preprocessed_biomarker_sequences.npy')
PHASE2=os.path.join(DRIVE,'Phase2_Collapse_Study'); RES=os.path.join(PHASE2,'results')
LAB_LOG=os.path.join(PHASE2,'phase2_lab_log.jsonl')
METRICS_DIR=os.path.join(PHASE2,'final_metrics'); os.makedirs(METRICS_DIR,exist_ok=True)
FEATURE_SIZE=48; VIT_DIM=768; NUM_CLASSES=3; SEED=42; IMG_SIZE=(96,96,96)
torch.manual_seed(SEED); np.random.seed(SEED)
print('config ready')


## 2. Data

In [ ]:
sp=np.load(SPLITS,allow_pickle=True)
pids_te,y_te=list(sp['pids_test']),list(sp['labels_test'])
pids_tr,y_tr=list(sp['pids_train']),list(sp['labels_train'])
pids_va,y_va=list(sp['pids_val']),list(sp['labels_val'])
clin_df=pd.read_csv(CSV)
FEATS=['AGE','PTGENDER','PTEDUCAT','APOE4','MMSE','ADAS13','RAVLT_immediate','RAVLT_learning','RAVLT_forgetting','FAQ']
patient_clin={p:torch.tensor(g[FEATS].values,dtype=torch.float32) for p,g in clin_df.groupby('PTID')}
bio_raw=np.load(BIO,allow_pickle=True); bio_obj=bio_raw.item() if (hasattr(bio_raw,'dtype') and bio_raw.dtype==object and bio_raw.ndim==0) else bio_raw
patient_bio={k:torch.tensor(np.asarray(v),dtype=torch.float32) for k,v in bio_obj.items()}
CLIN_DIM=len(FEATS); BIO_DIM=next(iter(patient_bio.values())).shape[-1]
logged={}
for line in open(LAB_LOG):
    line=line.strip()
    if line: e=json.loads(line); logged[e['condition']]=e['diagnostic']['ablation']['full']['acc']
print('data ready | test n =',len(pids_te),'| conditions in log:',len(logged))


## 3. ALL architecture variants (every class used across conditions)

In [ ]:
def make_swinvit(): return SwinUNETR(in_channels=1,out_channels=1,feature_size=FEATURE_SIZE,use_checkpoint=True).swinViT

class LSTM_long(nn.Module):  # lstm/fc/relu
    def __init__(self,ind,h=128,o=128):
        super().__init__(); self.lstm=nn.LSTM(ind,h,2,batch_first=True,dropout=0.2); self.fc=nn.Linear(h,o); self.relu=nn.ReLU()
    def forward(self,x): _,(h,_)=self.lstm(x); return self.relu(self.fc(h[-1]))
class LSTM_short(nn.Module):  # l/f/r
    def __init__(s,ind,h=128,o=128):
        super().__init__(); s.l=nn.LSTM(ind,h,2,batch_first=True,dropout=0.2); s.f=nn.Linear(h,o); s.r=nn.ReLU()
    def forward(s,x): _,(h,_)=s.l(x); return s.r(s.f(h[-1]))
class SwinEncWrap(nn.Module):  # encoder.swin.* + pool
    def __init__(self): super().__init__(); self.swin=make_swinvit(); self.pool=nn.AdaptiveAvgPool3d(1)
    def forward(self,x): return self.pool(self.swin(x)[-1]).flatten(1)

# A) C2/C3b/C3c: raw swinViT encoder, pool outside, img_proj, clin_branch/bio_branch, fusion_gate, classifier
class ModelA(nn.Module):
    def __init__(self,clin_dim,bio_dim,nc=3,unfreeze=False):
        super().__init__(); self.encoder=make_swinvit(); self.unfreeze_encoder=unfreeze
        for p in self.encoder.parameters(): p.requires_grad=unfreeze
        self.pool=nn.AdaptiveAvgPool3d(1)
        self.img_proj=nn.Sequential(nn.Linear(VIT_DIM,256),nn.ReLU(),nn.Linear(256,128))
        self.clin_branch=LSTM_long(clin_dim); self.bio_branch=LSTM_long(bio_dim)
        self.fusion_gate=nn.Sequential(nn.Linear(384,128),nn.ReLU(),nn.Linear(128,3),nn.Softmax(dim=1))
        self.classifier=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.ReLU(),nn.Dropout(0.5),nn.Linear(64,nc))
    def imgfeat(self,m):
        with torch.no_grad(): f=self.encoder(m.unsqueeze(1))[-1]
        return self.img_proj(self.pool(f).flatten(1))
    def forward(self,m,c,b):
        fi=self.imgfeat(m); fc=self.clin_branch(c); fb=self.bio_branch(b)
        st=torch.stack([fi,fc,fb],1); w=self.fusion_gate(torch.cat([fi,fc,fb],1))
        return self.classifier((st*w.unsqueeze(-1)).sum(1))

# B) C4: SwinEncWrap encoder, proj/cb/bb/gate/clf (short LSTM)
class ModelB(nn.Module):
    def __init__(s,cd,bd,nc=3):
        super().__init__(); s.encoder=SwinEncWrap()
        s.proj=nn.Sequential(nn.Linear(VIT_DIM,256),nn.ReLU(),nn.Linear(256,128))
        s.cb=LSTM_short(cd); s.bb=LSTM_short(bd)
        s.gate=nn.Sequential(nn.Linear(384,128),nn.ReLU(),nn.Linear(128,3),nn.Softmax(dim=1))
        s.clf=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.ReLU(),nn.Dropout(0.5),nn.Linear(64,nc))
    def imgfeat(s,m):
        with torch.no_grad(): f=s.encoder(m.unsqueeze(1))
        return s.proj(f)
    def forward(s,m,c,b):
        fi=s.imgfeat(m); fc=s.cb(c); fb=s.bb(b)
        st=torch.stack([fi,fc,fb],1); w=s.gate(torch.cat([fi,fc,fb],1))
        return s.clf((st*w.unsqueeze(-1)).sum(1))

# C) C5/C6: SwinEncWrap + proj/cb/bb + norms + gate_lin + log_temp + clf (+ optional aux_mri for C6)
class ModelC(nn.Module):
    def __init__(s,cd,bd,nc=3,aux=False):
        super().__init__(); s.encoder=SwinEncWrap()
        s.proj=nn.Sequential(nn.Linear(VIT_DIM,256),nn.ReLU(),nn.Linear(256,128))
        s.cb=LSTM_short(cd); s.bb=LSTM_short(bd)
        s.norm_img=nn.LayerNorm(128); s.norm_clin=nn.LayerNorm(128); s.norm_bio=nn.LayerNorm(128)
        s.gate_lin=nn.Sequential(nn.Linear(384,128),nn.ReLU(),nn.Linear(128,3))
        s.log_temp=nn.Parameter(torch.zeros(1))
        s.clf=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.ReLU(),nn.Dropout(0.5),nn.Linear(64,nc))
        if aux: s.aux_mri=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.ReLU(),nn.Linear(64,nc))
    def imgfeat(s,m):
        with torch.no_grad(): f=s.encoder(m.unsqueeze(1))
        return s.proj(f)
    def forward(s,m,c,b):
        fi=s.norm_img(s.imgfeat(m)); fc=s.norm_clin(s.cb(c)); fb=s.norm_bio(s.bb(b))
        st=torch.stack([fi,fc,fb],1); temp=torch.exp(s.log_temp).clamp(0.2,5.0)
        w=F.softmax(s.gate_lin(torch.cat([fi,fc,fb],1))/temp,dim=1)
        return s.clf((st*w.unsqueeze(-1)).sum(1))

# ARCH CANDIDATES: (name, builder). Loader tries each against each checkpoint.
ARCHES=[
  ('A_std',        lambda: ModelA(CLIN_DIM,BIO_DIM,unfreeze=False)),
  ('A_std_unfroz', lambda: ModelA(CLIN_DIM,BIO_DIM,unfreeze=True)),
  ('B_c4',         lambda: ModelB(CLIN_DIM,BIO_DIM)),
  ('C_gate',       lambda: ModelC(CLIN_DIM,BIO_DIM,aux=False)),
  ('C_gate_aux',   lambda: ModelC(CLIN_DIM,BIO_DIM,aux=True)),
]
print('architectures defined:',[a[0] for a in ARCHES])


## 4. Metrics + dataset + evaluator

In [ ]:
def gmean_cm(cm):
    g=[]
    for i in range(cm.shape[0]):
        tp=cm[i,i];fn=cm[i,:].sum()-tp;fp=cm[:,i].sum()-tp;tn=cm.sum()-(tp+fn+fp)
        se=tp/(tp+fn) if tp+fn>0 else 0; sp=tn/(tn+fp) if tn+fp>0 else 0; g.append((se*sp)**0.5)
    return float(np.mean(g))
def mauc(yt,pr):
    try: return float(roc_auc_score(yt,pr,multi_class='ovr',average='macro'))
    except Exception: return None
def all_metrics(yt,yp,pr):
    yt=np.array(yt);yp=np.array(yp); cm=confusion_matrix(yt,yp,labels=[0,1,2])
    return {'acc':float(accuracy_score(yt,yp)),'mcc':float(matthews_corrcoef(yt,yp)),'gmean':gmean_cm(cm),'auc':mauc(yt,pr)}
def boot_ci(yt,yp,pr,n=2000,seed=42):
    rng=np.random.default_rng(seed); yt=np.array(yt);yp=np.array(yp);pr=np.array(pr)
    A=[];M=[];G=[];U=[]; N=len(yt)
    for _ in range(n):
        idx=rng.integers(0,N,N)
        if len(np.unique(yt[idx]))<2: continue
        A.append(accuracy_score(yt[idx],yp[idx])); M.append(matthews_corrcoef(yt[idx],yp[idx]))
        G.append(gmean_cm(confusion_matrix(yt[idx],yp[idx],labels=[0,1,2])))
        u=mauc(yt[idx],pr[idx]);
        if u is not None: U.append(u)
    ci=lambda v:[float(np.percentile(v,2.5)),float(np.percentile(v,97.5))] if v else None
    return {'acc_ci':ci(A),'mcc_ci':ci(M),'gmean_ci':ci(G),'auc_ci':ci(U)}
def per_class(yt,yp):
    p,r,f,s=precision_recall_fscore_support(yt,yp,labels=[0,1,2],zero_division=0)
    return {c:{'precision':float(p[i]),'recall':float(r[i]),'f1':float(f[i]),'support':int(s[i])} for i,c in enumerate(['CN','MCI','Dementia'])}

etf=tio.Compose([tio.Resize(IMG_SIZE),tio.ZNormalization(masking_method=tio.ZNormalization.mean)])
class DS(Dataset):
    def __init__(s,pids,labels,d): s.pids=list(pids); s.y=torch.tensor(np.array(labels),dtype=torch.long); s.d=d
    def __len__(s): return len(s.pids)
    def __getitem__(s,i):
        p=s.pids[i]; vol=np.load(os.path.join(s.d,f'{p}_processed.npy'))
        subj=tio.Subject(mri=tio.ScalarImage(tensor=torch.tensor(vol,dtype=torch.float32).unsqueeze(0)))
        return etf(subj).mri.tensor.squeeze(0),patient_clin[p],patient_bio[p],s.y[i]
def coll(b):
    m,c,bi,y=zip(*b); return torch.stack(m),pad_sequence(c,batch_first=True),pad_sequence(bi,batch_first=True),torch.stack(y)

@torch.no_grad()
def evaluate(model,mri_dir):
    model.eval(); te=DataLoader(DS(pids_te,y_te,mri_dir),batch_size=8,shuffle=False,collate_fn=coll)
    P=[];T=[];PR=[]
    for m,c,b,y in te:
        lo=model(m.to(device),c.to(device),b.to(device)); PR.append(F.softmax(lo,1).cpu().numpy())
        P+=lo.argmax(1).cpu().tolist(); T+=y.tolist()
    PR=np.concatenate(PR); met=all_metrics(T,P,PR); met.update(boot_ci(T,P,PR))
    return {'metrics':met,'per_class':per_class(T,P),'confusion_matrix':confusion_matrix(T,P,labels=[0,1,2]).tolist(),'y_true':list(map(int,T)),'y_pred':list(map(int,P))}
print('evaluator ready')


## 5. Auto-match loader: try every architecture, keep what loads 100% AND matches the log
For each condition folder, tries all architectures on both preprocessing dirs; accepts the one that loads cleanly and reproduces the logged accuracy.

In [ ]:
CONDITION_DIRS={
  'condition2_monai_pretrained_frozen':     OLD_DIR,
  'condition3b_brain_preprocessed_frozen':  BRAIN_DIR,
  'condition3c_brain_finetuned':            BRAIN_DIR,
  'condition4_ssl_vicreg_frozen':           BRAIN_DIR,
  'condition4_ssl_mae_frozen':              BRAIN_DIR,
  'condition5_mae_improved_gate':           BRAIN_DIR,
  'condition5sweep_mae_cv_tuned_gate':      BRAIN_DIR,
  'condition6c_optuna_cv_forced_gate':      BRAIN_DIR,
}
# also auto-discover any other folders with best_model.pth
for d in os.listdir(RES):
    fp=os.path.join(RES,d,'best_model.pth')
    if os.path.exists(fp) and d not in CONDITION_DIRS: CONDITION_DIRS[d]=BRAIN_DIR

results={}
for cond,default_dir in CONDITION_DIRS.items():
    wpath=os.path.join(RES,cond,'best_model.pth')
    if not os.path.exists(wpath): print(f'{cond}: NO WEIGHTS -> use logged metrics only'); continue
    sd=torch.load(wpath,map_location='cpu')
    log_acc=logged.get(cond)
    best=None
    for aname,builder in ARCHES:
        for mri_dir in ([default_dir] if default_dir else [BRAIN_DIR,OLD_DIR]):
            try:
                model=builder().to(device)
                miss,unexp=model.load_state_dict(sd,strict=False)
                frac=1-len(miss)/len(model.state_dict())
                if frac<0.99: continue
                r=evaluate(model,mri_dir); acc=r['metrics']['acc']
                ok=(log_acc is None) or (abs(acc-log_acc)<0.02)
                if ok: best=(aname,mri_dir,r); raise StopIteration
            except StopIteration: raise
            except Exception: continue
        else: continue
        break
    if best is None:
        # try once more allowing dir flip for matching
        for aname,builder in ARCHES:
            for mri_dir in [BRAIN_DIR,OLD_DIR]:
                try:
                    model=builder().to(device); miss,_=model.load_state_dict(sd,strict=False)
                    if 1-len(miss)/len(model.state_dict())<0.99: continue
                    r=evaluate(model,mri_dir); acc=r['metrics']['acc']
                    if log_acc is None or abs(acc-log_acc)<0.02: best=(aname,mri_dir,r); break
                except Exception: continue
            if best: break
    if best is None:
        print(f'{cond}: could not verify any architecture -> use logged metrics only (likely overwritten checkpoint)')
        continue
    aname,mri_dir,r=best; m=r['metrics']
    print(f"{cond}: [{aname}] acc={m['acc']:.3f} (log {log_acc}) VERIFIED | mcc={m['mcc']:.3f} gmean={m['gmean']:.3f} auc={m['auc']:.3f}")
    print(f"   95% CI: acc {[round(x,3) for x in m['acc_ci']]}  mcc {[round(x,3) for x in m['mcc_ci']]}  auc {[round(x,3) for x in m['auc_ci']]}")
    results[cond]={**r,'arch':aname}
json.dump(results, open(os.path.join(METRICS_DIR,'final_metrics_all.json'),'w'), indent=2)
print(f'\n{len(results)} conditions VERIFIED with full metrics. Saved final_metrics_all.json')
print('Conditions without a recoverable model use their logged acc/MCC/G-mean (AUC/CI not available).')


## 6. Summary table (verified) + note unrecoverable ones

In [ ]:
print('%-42s %6s %6s %7s %6s  %-18s'%('condition','acc','mcc','gmean','auc','acc 95% CI'))
print('-'*100)
for cond,r in results.items():
    m=r['metrics']
    ci=m['acc_ci']
    print('%-42s %6.3f %6.3f %7.3f %6.3f  [%.3f, %.3f]'%(cond[:42],m['acc'],m['mcc'],m['gmean'],m['auc'],ci[0],ci[1]))
print('\nFrom lab log only (no recoverable model -> no AUC/CI):')
for cond,acc in logged.items():
    if cond not in results: print('  %-42s logged acc=%.3f (report acc/MCC/G-mean from log)'%(cond,acc))
